In [1]:
import pandas as pd
import sys
sys.path.insert(0, "..")
from utils import fill_empty_rows, import_datasets_to_db, create_file_path

In [2]:
# Read the .csv file
df_mazda = pd.read_csv(create_file_path("mazdaR.csv"))

In [3]:
# Check the df info
df_mazda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4484 entries, 0 to 4483
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ModType      3701 non-null   object
 1   HARDWARE     3701 non-null   object
 2   18881 P/N    2030 non-null   object
 3   Catch        3100 non-null   object
 4   12A650       2578 non-null   object
 5   EPC P/N      2284 non-null   object
 6   Mazda Model  3695 non-null   object
 7   Year         3671 non-null   object
 8   Engine       3637 non-null   object
 9   Trans        1785 non-null   object
 10  Emissions    1911 non-null   object
 11  SecType      1841 non-null   object
 12  Key          12 non-null     object
 13  Description  2777 non-null   object
 14  VIN EXAMPLE  337 non-null    object
 15  Unnamed: 15  50 non-null     object
dtypes: object(16)
memory usage: 560.6+ KB


In [58]:
# List with cols to keep
cols_to_keep = ['HARDWARE', 'ModType', '18881 P/N', '12A650', 'EPC P/N']

In [59]:
# Create a df with needed cols
df_mazda_filtered_cols = df_mazda[cols_to_keep]

In [60]:
# Remove rows where all cols have null (NaN) values,
# keeping only rows with at least one value (non-null) entry.
df_mazda_wo_nulls = df_mazda_filtered_cols[~df_mazda_filtered_cols.isnull().all(axis=1)]

### Delete all columns and keep only the part number related ones:
- ModType
- HARDWARE
- 18881 P/N
- 12A650
- EPC P/N

In [61]:
# Keep only rows where the cols have > 1 non-null values.
# The goal is to remove the lines where only col "18881 P/N" is filled out with a comment.
df_mazda_rm_one_line_notNull = df_mazda_wo_nulls[df_mazda_wo_nulls.notnull().sum(axis=1) != 1]

In [62]:
# Index list of rows filled out with the columns
index_list_repeated_cols = df_mazda_rm_one_line_notNull[df_mazda_rm_one_line_notNull['ModType'] == 'ModType'].index

In [63]:
# Drop rows with col names
df_mazda_dropped_cols = df_mazda_rm_one_line_notNull.drop(index=index_list_repeated_cols)

In [64]:
# Show the cols
df_mazda_dropped_cols.columns

Index(['HARDWARE', 'ModType', '18881 P/N', '12A650', 'EPC P/N'], dtype='object')

In [66]:
# Call the function to fill out null values
df_mazda_fill_nulls = fill_empty_rows(df_mazda_dropped_cols)

In [68]:
# Map to rename Hardware col
map_dict = {
    "HARDWARE": "hardware",
    "ModType":  "pn1",
    "18881 P/N": "pn2",
    "12A650": "pn3",
    "EPC P/N": "pn4"
}

In [69]:
# Rename hdw col
df_mazda_rn_hdw_col = df_mazda_fill_nulls.rename(columns=map_dict)

In [70]:
# Show the cols
df_mazda_rn_hdw_col.columns

Index(['hardware', 'pn1', 'pn2', 'pn3', 'pn4'], dtype='object')

In [71]:
# Check df info
df_mazda_rn_hdw_col.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3680 entries, 0 to 4483
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   hardware  3680 non-null   object
 1   pn1       3680 non-null   object
 2   pn2       3680 non-null   object
 3   pn3       3680 non-null   object
 4   pn4       3680 non-null   object
dtypes: object(5)
memory usage: 172.5+ KB


In [17]:
# Call the function to import the df cleaned to the postgreSQLdb
import_datasets_to_db(df_mazda_rn_hdw_col, 'part_numbers_mazda')

680